# 05 — Performance and Scaling

Why `iterrows`/`itertuples`/`.apply()` are slow (with numbers), memory-usage inspection, `MultiIndex` costs, merge performance, and the honest answer to "when does pandas stop being the right tool?" — a question every data engineer interview eventually asks.

In [1]:
import pandas as pd
import numpy as np
import time

N = 100_000
df = pd.DataFrame({
    "a": np.random.rand(N),
    "b": np.random.rand(N),
})

## 1. `iterrows` vs `itertuples` vs `.apply()` vs vectorized — measured

All four compute the same thing (`a + b`). The performance gap is large enough that reciting the *order* from memory (vectorized > `itertuples` > `.apply()` > `iterrows`) is a standard interview expectation, not just a nice-to-have.

In [2]:
def time_it(label, fn):
    start = time.perf_counter()
    fn()
    elapsed = time.perf_counter() - start
    print(f"{label:22s} {elapsed*1000:8.1f} ms")
    return elapsed

t_vectorized = time_it("vectorized (a + b)", lambda: df["a"] + df["b"])
t_apply = time_it("apply(axis=1)", lambda: df.apply(lambda row: row["a"] + row["b"], axis=1))
t_itertuples = time_it("itertuples loop", lambda: [row.a + row.b for row in df.itertuples()])
t_iterrows = time_it("iterrows loop", lambda: [row["a"] + row["b"] for _, row in df.iterrows()])

print(f"\niterrows is ~{t_iterrows/t_vectorized:.0f}x slower than vectorized")
print(f"apply is ~{t_apply/t_vectorized:.0f}x slower than vectorized")

vectorized (a + b)          0.8 ms


apply(axis=1)             314.3 ms
itertuples loop            22.1 ms


iterrows loop            1227.2 ms

iterrows is ~1453x slower than vectorized
apply is ~372x slower than vectorized


**Why the gap:** `iterrows()` reconstructs a `Series` (with its own dtype-unification and index overhead) for every row — the slowest option. `itertuples()` yields lightweight namedtuples — much cheaper, and the right choice on the rare occasion row-by-row Python logic is genuinely unavoidable. `.apply(axis=1)` sits between them. Vectorized operations avoid the Python-level per-row loop entirely.

## 2. Inspecting memory usage

`df.info(memory_usage="deep")` and `df.memory_usage(deep=True)` show actual bytes per column — `deep=True` matters for `object`-dtype columns, since the shallow default only counts pointer size, not the string data those pointers reference.

In [3]:
mixed = pd.DataFrame({
    "id": np.arange(50_000),
    "category": np.random.choice(["a", "b", "c"], 50_000),
})

print(mixed.memory_usage(deep=False))   # object column understated -- just pointer size
print()
print(mixed.memory_usage(deep=True))    # accurate -- includes actual string bytes

Index          132
id          400000
category    400000
dtype: int64

Index           132
id           400000
category    2900000
dtype: int64


## 3. `MultiIndex` — powerful, but not free

A `MultiIndex` (hierarchical row labels, e.g. after `groupby(["region", "rep"]).sum()`) is convenient for reshaping (`.unstack()`) but adds lookup overhead versus a flat `RangeIndex`, and `.loc[]` with partial tuple keys can be surprisingly slow if the index isn't sorted (`.sort_index()` first restores fast lookups). In a data pipeline, it's common to `.reset_index()` back to plain columns immediately after a `groupby` for anything downstream that doesn't need the hierarchical structure.

In [4]:
grouped = mixed.groupby(["category"]).agg(n=("id", "count"))
print(type(grouped.index))    # Index, not MultiIndex here (single group key)

multi = mixed.assign(bucket=mixed["id"] % 3).groupby(["category", "bucket"]).agg(n=("id", "count"))
print(type(multi.index))      # MultiIndex
multi.reset_index()            # flatten back to plain columns for downstream use

<class 'pandas.Index'>
<class 'pandas.MultiIndex'>


,category,bucket,n
0,a,0,5521
1,a,1,5505
2,a,2,5559
3,b,0,5580
4,b,1,5523
5,b,2,5577
6,c,0,5566
7,c,1,5639
8,c,2,5530


## 4. Merge performance

`pd.merge` on a plain column does an internal hash join — fine for most sizes. For **repeated** merges/lookups on the same key, setting that column as the index first (`df.set_index(key)`) and joining via `.join()` (which merges on the index) can be meaningfully faster, because pandas can reuse the index's hash structure instead of rebuilding it per call. Always verify assumed cardinality (`df[key].is_unique`) before merging — an unexpected many-to-many join silently blows up row counts and memory.

In [5]:
left = pd.DataFrame({"key": np.random.randint(0, 1000, 20_000), "val": np.random.rand(20_000)})
right = pd.DataFrame({"key": np.arange(1000), "lookup": np.random.rand(1000)}).set_index("key")

print("left key is unique:", left["key"].is_unique)   # False -- expect fan-out awareness

joined = left.join(right, on="key")   # index-based join on the pre-built right index
joined.head()

left key is unique: False


,key,val,lookup
0,211,0.380219,0.875375
1,859,0.557570,0.100537
2,465,0.093859,0.866877
3,535,0.454639,0.415857
4,696,0.909290,0.418342


## 5. When pandas stops being the right tool

Pandas is **single-machine, in-memory, single-threaded for most core operations** (though some operations release the GIL / use multiple cores under the hood in modern versions). Signals it's time to reach for something else:

- **Data doesn't fit in one machine's RAM even after `category`/downcasting/chunking** → Spark/Dask (distributed), or Polars (single-machine but much better memory efficiency and a lazy, multi-threaded execution engine).
- **A pipeline needs to scale horizontally / run on a cluster** → Spark — see `../../spark_practice/` for the equivalent operations (joins, window functions, groupBy) in a distributed engine; almost every pattern in these pandas notebooks has a direct Spark counterpart there.
- **CPU-bound, single-machine, but pandas itself is the bottleneck** → Polars or `pandas` with the `pyarrow` backend (`dtype_backend="pyarrow"`) often close most of the gap without changing the API much.

**Interview framing:** "Pandas is the right default for anything that fits comfortably in memory on one machine; the moment that stops being true, the fix is a different engine, not fighting pandas harder."

## 6. Interview Q&A

1. **"Rank `iterrows`, `itertuples`, `.apply()`, and a vectorized expression by speed."** — vectorized fastest, then `itertuples`, then `.apply()`, then `iterrows` slowest — `iterrows` rebuilds a full `Series` per row.
2. **"Your `df.memory_usage()` looks too small for a DataFrame full of strings — why?"** — the default `deep=False` only counts pointer size for `object` columns, not the referenced string data; use `deep=True`.
3. **"When would `.join()` beat `.merge()`?"** — when joining repeatedly on the same key — setting that key as the index once and using `.join()` reuses the index structure rather than rebuilding a hash join each call.
4. **"At what point would you stop using pandas for a pipeline?"** — once the data no longer fits comfortably in one machine's memory, or the job needs to scale across a cluster — that's a Spark/Dask problem, not a "tune pandas harder" problem.

## Summary

- Vectorize; if you must loop, `itertuples()` beats `iterrows()` and `.apply()`.
- Use `deep=True` to see real memory usage of `object` columns.
- `MultiIndex` is useful for reshaping but not free — flatten with `.reset_index()` when you don't need it downstream.
- Verify key uniqueness before merging; `.join()` on a pre-set index beats repeated `.merge()` calls on the same key.
- Pandas is single-machine/in-memory — beyond that, it's a Spark or Polars problem, not a pandas-tuning problem.
- Next: `06_interview_coding_problems.ipynb`.